# CNN-Based Brain Tumor Detection System
### Fast GPU Training & Evaluation in Google Colab or Kaggle

This notebook trains and evaluates the 4-class MRI brain tumor classification model on a free GPU (NVIDIA T4).

**Classes:** Glioma, Meningioma, Pituitary, No Tumor

---

## 1. Verify GPU Hardware Acceleration
In Colab: `Runtime` &rarr; `Change runtime type` &rarr; `T4 GPU`.
In Kaggle: `Settings` &rarr; `Accelerator` &rarr; `GPU T4 x2`.

In [ ]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"SUCCESS: Found {len(gpus)} GPU(s):", gpus)
else:
    print("WARNING: No GPU detected. Please enable GPU accelerator in runtime settings.")

## 2. Clone the Repository
Clone the project repository into the working environment.

In [ ]:
import os
from pathlib import Path

# Clone repo if not already cloned
if not Path("cnn-brain-tumor-detection").is_dir():
    !git clone https://github.com/Biswa554/cnn-brain-tumor-detection.git

%cd cnn-brain-tumor-detection

## 3. Prepare the Dataset
- **If in Kaggle:** The dataset is mounted automatically at `/kaggle/input/brain-tumor-mri-dataset`.
- **If in Google Colab:** Download using the Kaggle API or upload your dataset.

In [ ]:
import os
from pathlib import Path

kaggle_dataset_path = Path("/kaggle/input/brain-tumor-mri-dataset")

if kaggle_dataset_path.exists():
    print("Kaggle environment detected. Symlinking dataset...")
    !mkdir -p datasets
    !ln -sf /kaggle/input/brain-tumor-mri-dataset/Training datasets/Training
    !ln -sf /kaggle/input/brain-tumor-mri-dataset/Testing datasets/Testing
else:
    print("Google Colab / External environment detected.")
    # If datasets/ doesn't exist, download from Kaggle API:
    if not Path("datasets/Training").exists():
        # Provide your kaggle.json or set environment variables if needed:
        # os.environ['KAGGLE_USERNAME'] = 'your_username'
        # os.environ['KAGGLE_KEY'] = 'your_key'
        !pip install -q kaggle
        !kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset
        !unzip -q brain-tumor-mri-dataset.zip -d datasets

# Verify folder structure
from src.config import TRAIN_DIR, TEST_DIR
from src.data_preprocessing import inspect_split
print("Training images:", inspect_split(TRAIN_DIR, "Training"))
print("Testing images:", inspect_split(TEST_DIR, "Testing"))

## 4. Install Dependencies

In [ ]:
!pip install -q -r requirements.txt

## 5. Train the CNN Model on GPU
Trains the CNN with early stopping and automatic best-model checkpointing.
On a T4 GPU, 20 epochs take only ~1 to 2 minutes!

In [ ]:
!python src/train.py

## 6. Evaluate on the Unseen Testing Split
Evaluates precision, recall, F1-score, and confusion matrix.

In [ ]:
!python src/evaluate.py

# Display Confusion Matrix directly in notebook
from IPython.display import Image, display
if Path("results/confusion_matrix.png").exists():
    display(Image("results/confusion_matrix.png"))

## 7. Test Single Image Prediction

In [ ]:
!python src/predict.py --image datasets/Testing/glioma/Te-gl_10.jpg

## 8. Download Trained Model to Run Streamlit Locally
Download the `.keras` model file to your local computer and place it in your local `models/` directory.
Then run `streamlit run app.py` on your machine!

In [ ]:
try:
    from google.colab import files
    files.download("models/brain_tumor_cnn.keras")
    print("Model download triggered!")
except ImportError:
    print("In Kaggle: Look at the right sidebar 'Output' section to download brain_tumor_cnn.keras.")